# Experimentação

Este notebook orquestra a Fase 1 e 2 da etapa de experimentação, seguindo o protocolo descrito em `docs/experimentation.md`.

## Objetivos

## Setups e Imports

In [1]:
import sys
from pathlib import Path

# Garante que a raiz do projeto esta no sys.path
ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
        ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate project root.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Manipulacao de Dados
import pandas as pd
import numpy as np

# Visualizacao de Dados
import matplotlib.pyplot as plt
import seaborn as sns

# Pre-processamento e Modelagem
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    RandomizedSearchCV,
)
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Metricas de Avaliacao
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    make_scorer,
)

# Modelos de Machine Learning
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from src.utils.exp import (
    MLPClassifierWrapper,
    build_k_grid,
    evaluate_round3_model_strategies,
    extract_selected_feature_names,
    format_selected_features_log,
    get_processed_feature_names,
    summarize_grid_search_results,
)

# importando os transformers customizados
from src.features.geo_transformer import GeoTransformer
from src.features.feature_engineer_transformer import FeatureEngineerTransformer
from src.utils.logging_config import get_logger

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# MLflow
# ATENCAO: o servidor do MLflow precisa estar rodando antes de executar este notebook.
# Em um terminal separado, execute:
#   mlflow server --host 127.0.0.1 --port 5000
# Sem isso, as celulas de tracking falharao com ConnectionRefusedError.
import hashlib
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://localhost:5000")

## Processamento dos Dados

In [2]:
# dataload
df = pd.read_excel('../data/raw/Telco_customer_churn.xlsx')
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [3]:
# Informações gerais sobre o dataset
print("=== INFORMAÇÕES GERAIS DO DATASET ===\n")
print(df.info())

# Shape
print("\n=== SHAPE DO DATASET ===")
print(f"Linhas: {df.shape[0]}, Colunas: {df.shape[1]}")

# Colunas
print("\n=== COLUNAS DO DATASET ===")
print(df.columns.tolist())

=== INFORMAÇÕES GERAIS DO DATASET ===

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null 

In [4]:
# Transformação da coluna 'Total Charges' para numérica, tratando erros e preenchendo valores ausentes com 0.
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
df['Total Charges'] = df['Total Charges'].fillna(0)

In [5]:
# dropando colunas irrelevantes para a modelagem
drop_cols = [
    'Country',
    'State',
    'Lat Long',
    'Churn Label',
    'Churn Reason',
    'Count'
]


df.drop(columns=drop_cols, inplace=True)

In [6]:
target = "Churn Value"
meta_cols = ["CLTV", "CustomerID"]

feature_cols = [
    col for col in df.columns
    if col not in [target] + meta_cols
]

X = df[feature_cols]
y = df[target]

metadata = df[meta_cols]


## Splits e Validação

In [7]:
# Protocolo de validação cruzada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# 
X_train_val, X_test, y_train_val, y_test, metadata_train_val, metadata_test = train_test_split(
    X,
    y,
    metadata,
    test_size=0.3,
    stratify=y,
    random_state=42
)

# Checando os Splits
print("=== SPLITS ===")
print(f"Treino/Validação: {X_train_val.shape[0]} amostras")
print(f"Teste: {X_test.shape[0]} amostras")

# Checando a distribuição da variável alvo nos splits
print("\n=== DISTRIBUIÇÃO DA VARIÁVEL ALVO NOS SPLITS ===")
print("Treino/Validação:")
print(y_train_val.value_counts(normalize=True))
print("\nTeste:")
print(y_test.value_counts(normalize=True))

# Checando o formato dos dados
print("\n=== FORMATO DOS DADOS ===")
print(f"X_train_val: {X_train_val.shape}")
print(f"y_train_val: {y_train_val.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")


=== SPLITS ===
Treino/Validação: 4930 amostras
Teste: 2113 amostras

=== DISTRIBUIÇÃO DA VARIÁVEL ALVO NOS SPLITS ===
Treino/Validação:
Churn Value
0    0.734686
1    0.265314
Name: proportion, dtype: float64

Teste:
Churn Value
0    0.734501
1    0.265499
Name: proportion, dtype: float64

=== FORMATO DOS DADOS ===
X_train_val: (4930, 24)
y_train_val: (4930,)
X_test: (2113, 24)
y_test: (2113,)


## Baseline Inicial

- Treina Dummy e Regressão Logística e compara com a MLP e outros modelos de Árvores

In [8]:
# Definindo a etapa de pré-processamento para variáveis categóricas com OHE e numéricas com passthrough
ohe = OneHotEncoder(handle_unknown="ignore")
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", ohe, make_column_selector(dtype_include=["object", "category"])),
        ("num", "passthrough", make_column_selector(dtype_exclude=["object", "category"])),
    ],
    remainder="drop",
)

# Dicionário de Pipelines para cada modelo
baseline_params = dict(
    drop_churn_score=True,
    add_engagement_score=False,
    add_tenure_group=False,
    add_tenure_log=False,
    add_contract_ordinal=False,
    add_family_stability=False,
    add_fiber_no_support=False,
    add_support_gap_count=False,
    add_payment_automatic_flag=False,
    add_electronic_check_flag=False,
    add_paperless_echeck_flag=False,
    add_price_pressure_ratio=False,
)

models = {
    "Dummy": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", DummyClassifier(strategy="most_frequent")),
    ]),
    "LogisticRegression": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ]),
    "DecisionTree": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", DecisionTreeClassifier(
            random_state=42,
            class_weight="balanced",
        )),
    ]),
    "RandomForest": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", RandomForestClassifier(
            random_state=42,
            n_jobs=-1,
            class_weight="balanced_subsample",
        )),
    ]),
    "XGBoost": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    "MLP": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", MLPClassifierWrapper(
            hidden_dim=64,
            batch_size=64,
            lr=1e-3,
            weight_decay=1e-5,
            max_epochs=80,
            patience=8,
            val_size=0.15,
            threshold=0.5,
            random_state=42,
            verbose=False,
        )),
    ]),
}


In [9]:
# definindo o scoring para avaliação dos modelos
scoring = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "recall": make_scorer(recall_score, zero_division=0),
    "precision": make_scorer(precision_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
}

In [10]:
metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]
rows = []
fold_results = {}

for model_name, estimator in models.items():
    print(f"=== AVALIANDO MODELO: {model_name} ===")
    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    fold_results[model_name] = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{m: cv_res[f"test_{m}"] for m in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    rows.append({
        "model": model_name,
        **{f"{m}_mean": cv_res[f"test_{m}"].mean() for m in metrics},
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    })

results_cv = (
    pd.DataFrame(rows)
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===")
display(results_cv.round(4))

=== AVALIANDO MODELO: Dummy ===
=== AVALIANDO MODELO: LogisticRegression ===
=== AVALIANDO MODELO: DecisionTree ===
=== AVALIANDO MODELO: RandomForest ===
=== AVALIANDO MODELO: XGBoost ===
=== AVALIANDO MODELO: MLP ===

=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===


,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,LogisticRegression,0.6780,0.8575,0.8142,0.5314,0.6430,0.0232,0.0112
1,MLP,0.6730,0.8562,0.8012,0.5303,0.6380,0.9319,0.0135
2,XGBoost,0.6492,0.8438,0.6743,0.5685,0.6167,0.0599,0.0156
3,RandomForest,0.6323,0.8393,0.5091,0.6467,0.5692,0.1569,0.0607
4,DecisionTree,0.3927,0.6684,0.5099,0.5147,0.5120,0.0222,0.0111
5,Dummy,0.2653,0.5000,0.0000,0.0000,0.0000,0.0099,0.0106


### Validando o Wrapper

- Aplicação da MLP fora do pipeline para validação dos resultados do Wrapper.

In [11]:
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
)

from src.models.mlp import MLP, evaluate, train_with_early_stopping

# ---------- Config ----------
MLP_EPOCHS = 80
MLP_BATCH_SIZE = 64
MLP_LR = 1e-3
MLP_WD = 1e-5
MLP_HIDDEN_DIM = 64
MLP_DROPOUT = 0.0
MLP_THRESHOLD = 0.5

ES_PATIENCE = int(np.ceil(0.2 * MLP_EPOCHS))
ES_MIN_DELTA = 1e-3
ES_VAL_SIZE = 0.15

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _rows(X, idx):
    return X.iloc[idx] if hasattr(X, "iloc") else X[idx]


def _to_dense_float32(x):
    if sp.issparse(x):
        x = x.toarray()
    return np.asarray(x, dtype=np.float32)


baseline_fe = FeatureEngineerTransformer(**baseline_params)
baseline_geo = GeoTransformer(strategy="drop")

mlp_manual_folds = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train_val, y_train_val), start=1):
    fit_start = time.perf_counter()

    X_tr_raw = _rows(X_train_val, tr_idx)
    X_va_raw = _rows(X_train_val, va_idx)
    y_tr = np.asarray(_rows(y_train_val, tr_idx), dtype=np.float32)
    y_va = np.asarray(_rows(y_train_val, va_idx), dtype=np.float32)

    X_tr_base = baseline_fe.fit_transform(X_tr_raw, y_tr)
    X_tr_base = baseline_geo.fit_transform(X_tr_base, y_tr)
    X_va_base = baseline_fe.transform(X_va_raw)
    X_va_base = baseline_geo.transform(X_va_base)

    prep_fold = clone(preprocessor)
    X_tr_enc = prep_fold.fit_transform(X_tr_base, y_tr)
    X_va_enc = prep_fold.transform(X_va_base)

    idx_all = np.arange(len(y_tr))
    idx_tr, idx_es = train_test_split(
        idx_all,
        test_size=ES_VAL_SIZE,
        stratify=y_tr,
        random_state=42 + fold,
    )

    X_tr_fit = X_tr_enc[idx_tr]
    y_tr_fit = y_tr[idx_tr]
    X_tr_es = X_tr_enc[idx_es]
    y_tr_es = y_tr[idx_es]

    scaler = StandardScaler(with_mean=False)
    X_tr_fit = scaler.fit_transform(X_tr_fit)
    X_tr_es = scaler.transform(X_tr_es)
    X_va_sc = scaler.transform(X_va_enc)

    X_tr_fit = _to_dense_float32(X_tr_fit)
    X_tr_es = _to_dense_float32(X_tr_es)
    X_va_sc = _to_dense_float32(X_va_sc)

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(
            torch.tensor(X_tr_fit, dtype=torch.float32),
            torch.tensor(y_tr_fit, dtype=torch.float32),
        ),
        batch_size=MLP_BATCH_SIZE,
        shuffle=True,
    )
    es_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(
            torch.tensor(X_tr_es, dtype=torch.float32),
            torch.tensor(y_tr_es, dtype=torch.float32),
        ),
        batch_size=MLP_BATCH_SIZE,
        shuffle=False,
    )

    model = MLP(
        input_dim=X_tr_fit.shape[1],
        hidden_dim=MLP_HIDDEN_DIM,
        output_dim=1,
        dropout=MLP_DROPOUT,
    ).to(DEVICE)

    pos = float((y_tr_fit == 1).sum())
    neg = float((y_tr_fit == 0).sum())
    pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=MLP_LR, weight_decay=MLP_WD)

    epochs_trained = train_with_early_stopping(
        model,
        train_loader,
        es_loader,
        optimizer,
        criterion,
        device=DEVICE,
        max_epochs=MLP_EPOCHS,
        patience=ES_PATIENCE,
        min_delta=ES_MIN_DELTA,
        threshold=MLP_THRESHOLD,
    )
    best_es_loss, _ = evaluate(
        model,
        es_loader,
        criterion,
        device=DEVICE,
        threshold=MLP_THRESHOLD,
    )
    fit_time_s = time.perf_counter() - fit_start

    score_start = time.perf_counter()
    X_va_t = torch.tensor(X_va_sc, dtype=torch.float32).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(X_va_t).squeeze(1)
        prob = torch.sigmoid(logits).cpu().numpy()

    pred = (prob >= MLP_THRESHOLD).astype(int)

    mlp_manual_folds.append(
        {
            "fold": fold,
            "epochs_trained": epochs_trained,
            "best_es_loss": best_es_loss,
            "pr_auc": average_precision_score(y_va, prob),
            "roc_auc": roc_auc_score(y_va, prob),
            "recall": recall_score(y_va, pred, zero_division=0),
            "precision": precision_score(y_va, pred, zero_division=0),
            "f1": f1_score(y_va, pred, zero_division=0),
            "fit_time_s": fit_time_s,
            "score_time_s": time.perf_counter() - score_start,
        }
    )

mlp_fold_results = pd.DataFrame(mlp_manual_folds)

mlp_cv_summary = pd.DataFrame(
    [
        {
            "model": "MLP_manual",
            "pr_auc_mean": mlp_fold_results["pr_auc"].mean(),
            "roc_auc_mean": mlp_fold_results["roc_auc"].mean(),
            "recall_mean": mlp_fold_results["recall"].mean(),
            "precision_mean": mlp_fold_results["precision"].mean(),
            "f1_mean": mlp_fold_results["f1"].mean(),
            "fit_time_mean_s": mlp_fold_results["fit_time_s"].mean(),
            "score_time_mean_s": mlp_fold_results["score_time_s"].mean(),
        }
    ]
)

display(mlp_cv_summary.round(4))

,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,MLP_manual,0.6616,0.8515,0.8035,0.5282,0.6363,0.9725,0.0037


### Conclusão

Na tabela de baselines, a `LogisticRegression` apresentou o melhor desempenho geral, com `PR-AUC = 0.6782` e `ROC-AUC = 0.8576`, ficando levemente acima da `MLP` (`PR-AUC = 0.6728` e `ROC-AUC = 0.8551`). Ainda assim, a diferenÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â§a entre os dois modelos foi pequena, e a `MLP` superou o benchmark de Árvore selecionado nesta rodada, o `XGBoost` (`PR-AUC = 0.6492`). Isso indica que, já na base original, a MLP se mostrou competitiva em relação ao baseline linear e ao benchmark não linear.

Na etapa de validação do Wrapper do MLP, a `MLP` dentro do `Pipeline` manteve desempenho consistente e até ligeiramente superior a versão manual fora do pipeline. Com isso, o wrapper foi validado com sucesso para uso no fluxo de experimentação, trazendo a vantagem de encapsular o preprocessamento e validação cruzada dentro da mesma estrutura, com menor risco de leakage e maior facilidade para evoluir o pipeline com feature engineering e seleção de features.


### Logging no MLflow

## Feature Engineering

**Objetivo**: 
- Adicionar poder preditivo aos modelos de forma controlada

### Round 1 - FE orientada a hipótese

- adicionando features a partir de hipóteses construídas a partir da EDA;

In [12]:
round1_fe_params = dict(
    drop_churn_score=True,
    add_engagement_score=True,
    add_tenure_group=True,
    add_tenure_log=True,
    add_contract_ordinal=True,
    add_family_stability=True,
    add_fiber_no_support=True,
    add_support_gap_count=True,
    add_payment_automatic_flag=True,
    add_electronic_check_flag=True,
    add_paperless_echeck_flag=True,
    add_price_pressure_ratio=True,
)


models = {
    "LogisticRegression": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round1_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ]),
    "XGBoost": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round1_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    "MLP": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round1_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", MLPClassifierWrapper(
            hidden_dim=64,
            batch_size=64,
            lr=1e-3,
            weight_decay=1e-5,
            max_epochs=80,
            patience=8,
            val_size=0.15,
            threshold=0.5,
            random_state=42,
            verbose=False,
        )),
    ]),
}

In [13]:
metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]
rows = []
fold_results = {}

for model_name, estimator in models.items():
    print(f"=== AVALIANDO MODELO: {model_name} ===")

    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    fold_results[model_name] = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{metric: cv_res[f"test_{metric}"] for metric in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    rows.append({
        "model": model_name,
        **{f"{metric}_mean": cv_res[f"test_{metric}"].mean() for metric in metrics},
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    })

results_cv = (
    pd.DataFrame(rows)
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===")
display(results_cv.round(4))

=== AVALIANDO MODELO: LogisticRegression ===
=== AVALIANDO MODELO: XGBoost ===
=== AVALIANDO MODELO: MLP ===

=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===


,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,LogisticRegression,0.6893,0.8625,0.8234,0.5382,0.6508,0.0546,0.0211
1,MLP,0.6853,0.8604,0.8165,0.5344,0.6452,0.6744,0.0260
2,XGBoost,0.6465,0.8418,0.6636,0.5752,0.6160,0.0659,0.0269


### Round 2 - Adicionando `Churn Score`

- Verificando o ganho com stacking de outro modelo;

In [14]:
round2_fe_params = dict(
    drop_churn_score=False,
    add_engagement_score=True,
    add_tenure_group=True,
    add_tenure_log=True,
    add_contract_ordinal=True,
    add_family_stability=True,
    add_fiber_no_support=True,
    add_support_gap_count=True,
    add_payment_automatic_flag=True,
    add_electronic_check_flag=True,
    add_paperless_echeck_flag=True,
    add_price_pressure_ratio=True,
)


models = {
    "LogisticRegression": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round2_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ]),
    "XGBoost": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round2_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    "MLP": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round2_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", MLPClassifierWrapper(
            hidden_dim=64,
            batch_size=64,
            lr=1e-3,
            weight_decay=1e-5,
            max_epochs=80,
            patience=8,
            val_size=0.15,
            threshold=0.5,
            random_state=42,
            verbose=False,
        )),
    ]),
}

In [15]:
metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]
rows = []
fold_results = {}

for model_name, estimator in models.items():
    print(f"=== AVALIANDO MODELO: {model_name} ===")

    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    fold_results[model_name] = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{metric: cv_res[f"test_{metric}"] for metric in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    rows.append({
        "model": model_name,
        **{f"{metric}_mean": cv_res[f"test_{metric}"].mean() for metric in metrics},
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    })

results_cv = (
    pd.DataFrame(rows)
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===")
display(results_cv.round(4))

=== AVALIANDO MODELO: LogisticRegression ===
=== AVALIANDO MODELO: XGBoost ===
=== AVALIANDO MODELO: MLP ===

=== RESULTADOS MÉDIOS DA VALIDAÇÃO CRUZADA ===


,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,XGBoost,0.9512,0.9802,0.8738,0.8404,0.8566,0.0666,0.0270
1,LogisticRegression,0.9393,0.9759,0.9297,0.7827,0.8497,0.0440,0.0203
2,MLP,0.9355,0.9743,0.9396,0.7492,0.8332,1.0523,0.0236


### Round 3 - Testando encoding para City

Definindo as mesmas features para todos as estratégias de encoding

In [16]:
round3_fe_params = dict(
    drop_churn_score=False,
    add_engagement_score=True,
    add_tenure_group=True,
    add_tenure_log=True,
    add_contract_ordinal=True,
    add_family_stability=True,
    add_fiber_no_support=True,
    add_support_gap_count=True,
    add_payment_automatic_flag=True,
    add_electronic_check_flag=True,
    add_paperless_echeck_flag=True,
    add_price_pressure_ratio=True,
)


Configuração das estratégias por modelo

In [17]:
assert "City" in X_train_val.columns, "City precisa estar presente em X_train_val para o Round 3."

round3_metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]

round3_strategy_specs = {
    "LogisticRegression": [
        ("frequency", "LogisticRegression_Frequency"),
        ("target", "LogisticRegression_Target"),
        ("geo_cluster", "LogisticRegression_Geo_Cluster"),
        ("zip_region", "LogisticRegression_ZIP"),
        ("risk_band", "LogisticRegression_Risk_Band"),
    ],
    "XGBoost": [
        ("frequency", "XGBoost_Frequency"),
        ("target", "XGBoost_Target"),
        ("geo_cluster", "XGBoost_Geo_Cluster"),
        ("zip_region", "XGBoost_ZIP"),
        ("risk_band", "XGBoost_Risk_Band"),
    ],
    "MLP": [
        ("frequency", "MLP_Frequency"),
        ("target", "MLP_Target"),
        ("geo_cluster", "MLP_Geo_Cluster"),
        ("zip_region", "MLP_ZIP"),
        ("risk_band", "MLP_Risk_Band"),
        ("city_embedding", "MLP_CityEmbedding"),
    ],
}

round3_logistic_params = {
    "max_iter": 1000,
    "class_weight": "balanced",
    "random_state": 42,
}

round3_xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "scale_pos_weight": (y_train_val == 0).sum() / (y_train_val == 1).sum(),
    "random_state": 42,
    "n_jobs": -1,
}

round3_mlp_params = {
    "hidden_dim": 64,
    "batch_size": 64,
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "max_epochs": 80,
    "patience": 8,
    "val_size": 0.15,
    "threshold": 0.5,
    "random_state": 42,
    "verbose": False,
}

round3_embedding_params = {
    "city_column": "City",
    "geo_drop_columns": ("Zip Code", "Latitude", "Longitude", "Lat Long"),
    "embedding_dim": None,
}

In [18]:
round3_results_by_model = {}
round3_fold_results_by_model = {}

for model_name, strategy_specs in round3_strategy_specs.items():
    print(f"=== ROUND 3: {model_name} ===")

    model_results, model_fold_results = evaluate_round3_model_strategies(
        model_name,
        strategy_specs,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        preprocessor=preprocessor,
        fe_params=round3_fe_params,
        y_reference=y_train_val,
        metrics=round3_metrics,
        target_smoothing=20.0,
        logistic_params=round3_logistic_params,
        xgb_params=round3_xgb_params,
        mlp_params=round3_mlp_params,
        embedding_params=round3_embedding_params,
    )

    round3_results_by_model[model_name] = model_results
    round3_fold_results_by_model[model_name] = model_fold_results

=== ROUND 3: LogisticRegression ===
=== ROUND 3: XGBoost ===
=== ROUND 3: MLP ===


Resultados por modelo: LogisticRegression

In [19]:
display(round3_results_by_model["LogisticRegression"].round(4))

,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s,base_model,strategy
0,LogisticRegression_ZIP,0.9394,0.9759,0.9312,0.7835,0.8508,0.0610,0.0212,LogisticRegression,zip_region
1,LogisticRegression_Geo_Cluster,0.9390,0.9757,0.9266,0.7842,0.8493,0.3292,0.0241,LogisticRegression,geo_cluster
2,LogisticRegression_Frequency,0.9390,0.9758,0.9266,0.7807,0.8472,0.0466,0.0212,LogisticRegression,frequency
3,LogisticRegression_Target,0.9221,0.9689,0.8777,0.7792,0.8253,0.0768,0.0199,LogisticRegression,target
4,LogisticRegression_Risk_Band,0.9147,0.9644,0.8716,0.7712,0.8181,0.0420,0.0203,LogisticRegression,risk_band


Resultados por modelo: XGBoost

In [20]:
display(round3_results_by_model["XGBoost"].round(4))

,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s,base_model,strategy
0,XGBoost_Geo_Cluster,0.9527,0.9809,0.8838,0.8468,0.8648,0.0968,0.0357,XGBoost,geo_cluster
1,XGBoost_ZIP,0.9515,0.9808,0.8823,0.8493,0.8654,0.0975,0.0304,XGBoost,zip_region
2,XGBoost_Frequency,0.9493,0.9799,0.8776,0.8478,0.8623,0.0683,0.0275,XGBoost,frequency
3,XGBoost_Risk_Band,0.9356,0.9731,0.8494,0.8374,0.8432,0.0718,0.0282,XGBoost,risk_band
4,XGBoost_Target,0.9330,0.9719,0.8234,0.8559,0.8392,0.0702,0.0276,XGBoost,target


Resultados por modelo: MLP

In [21]:
display(round3_results_by_model["MLP"].round(4))

,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s,base_model,strategy
0,MLP_ZIP,0.9376,0.9751,0.9319,0.7673,0.8412,1.1799,0.0293,MLP,zip_region
1,MLP_Frequency,0.9375,0.9754,0.9342,0.7701,0.8439,1.0941,0.0249,MLP,frequency
2,MLP_Geo_Cluster,0.9373,0.9751,0.9266,0.7675,0.8387,1.2197,0.0313,MLP,geo_cluster
3,MLP_CityEmbedding,0.9245,0.9689,0.9174,0.7469,0.8226,1.1935,0.0270,MLP,city_embedding
4,MLP_Target,0.9189,0.9679,0.9006,0.7668,0.8277,1.2915,0.0262,MLP,target
5,MLP_Risk_Band,0.9084,0.9616,0.8662,0.7671,0.8127,1.3845,0.0285,MLP,risk_band


Resultado consolidado do Round 3

In [22]:
round3_results_all = (
    pd.concat(round3_results_by_model.values(), ignore_index=True)
    .sort_values(["base_model", "pr_auc_mean"], ascending=[True, False])
    .reset_index(drop=True)
)

display(round3_results_all.round(4))

,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s,base_model,strategy
0,LogisticRegression_ZIP,0.9394,0.9759,0.9312,0.7835,0.8508,0.0610,0.0212,LogisticRegression,zip_region
1,LogisticRegression_Geo_Cluster,0.9390,0.9757,0.9266,0.7842,0.8493,0.3292,0.0241,LogisticRegression,geo_cluster
2,LogisticRegression_Frequency,0.9390,0.9758,0.9266,0.7807,0.8472,0.0466,0.0212,LogisticRegression,frequency
3,LogisticRegression_Target,0.9221,0.9689,0.8777,0.7792,0.8253,0.0768,0.0199,LogisticRegression,target
4,LogisticRegression_Risk_Band,0.9147,0.9644,0.8716,0.7712,0.8181,0.0420,0.0203,LogisticRegression,risk_band
5,MLP_ZIP,0.9376,0.9751,0.9319,0.7673,0.8412,1.1799,0.0293,MLP,zip_region
6,MLP_Frequency,0.9375,0.9754,0.9342,0.7701,0.8439,1.0941,0.0249,MLP,frequency
7,MLP_Geo_Cluster,0.9373,0.9751,0.9266,0.7675,0.8387,1.2197,0.0313,MLP,geo_cluster
8,MLP_CityEmbedding,0.9245,0.9689,0.9174,0.7469,0.8226,1.1935,0.0270,MLP,city_embedding
9,MLP_Target,0.9189,0.9679,0.9006,0.7668,0.8277,1.2915,0.0262,MLP,target


### Conclusão

As variáveis geográficas não demonstraram ganho robusto o suficiente para justificar sua incorporação no pipeline final. Embora algumas estratégias, como zip_region na Regressão Logística e geo_cluster no XGBoost, tenham produzido pequenas melhoras marginais, os ganhos foram muito discretos e inconsistentes entre os modelos. Na MLP, inclusive, abordagens mais sofisticadas como CityEmbedding elevaram o recall, mas com perda relevante de precisão. Considerando o aumento de complexidade, o risco de instabilidade e o baixo retorno incremental observado, a decisão é não incluir as variáveis geográficas na versão final do conjunto de features.

### Round 4 - Feature Selection Exploratoria

Objetivo desta rodada:
- explorar faixas promissoras de `K` para `LogisticRegression`, `XGBoost` e `MLP`;
- comparar `f_classif` e `mutual_info_classif` sem `GridSearchCV`;
- consolidar o ranking por `PR-AUC > ROC-AUC > Recall`.

In [ ]:
assert "City" in X_train_val.columns, "City precisa estar presente em X_train_val para esta rodada."
assert "Churn Score" in X_train_val.columns, "Churn Score precisa estar presente em X_train_val para esta rodada."
assert "CLTV" not in X_train_val.columns, "CLTV deve permanecer fora de X_train_val como metadata."

round4_logger = get_logger("round4_feature_selection")
round4_sort_columns = ["pr_auc_mean", "roc_auc_mean", "recall_mean"]
round4_sort_ascending = [False, False, False]


def rank_round4_results(df):
    return df.sort_values(round4_sort_columns, ascending=round4_sort_ascending).reset_index(drop=True)


selector_label_map = {
    f_classif: "f_classif",
    mutual_info_classif: "mutual_info_classif",
}
selector_specs = [
    (f_classif, "f_classif"),
    (mutual_info_classif, "mutual_info_classif"),
]

round4_fe_params = dict(
    drop_churn_score=False,
    add_engagement_score=True,
    add_tenure_group=True,
    add_tenure_log=True,
    add_contract_ordinal=True,
    add_family_stability=True,
    add_fiber_no_support=True,
    add_support_gap_count=True,
    add_payment_automatic_flag=True,
    add_electronic_check_flag=True,
    add_paperless_echeck_flag=True,
    add_price_pressure_ratio=True,
)

round4_fe = FeatureEngineerTransformer(**round4_fe_params)
round4_geo = GeoTransformer(strategy="drop")

X_train_val_round4 = round4_fe.fit_transform(X_train_val, y_train_val)
X_train_val_round4 = round4_geo.fit_transform(X_train_val_round4, y_train_val)

processed_feature_names = get_processed_feature_names(
    preprocessor,
    X_train_val_round4,
    y_train_val,
)

k_grid = build_k_grid(
    n_features_processed=len(processed_feature_names),
    min_k=10,
    include_all=False,
    step=10,
)

round4_logger.info(
    "Round 4 | total de features processadas=%d | k_grid=%s",
    len(processed_feature_names),
    k_grid,
)


def build_round4_estimators():
    return {
        "LogisticRegression": Pipeline(
            [
                ("fe", FeatureEngineerTransformer(**round4_fe_params)),
                ("geo", GeoTransformer(strategy="drop")),
                ("prep", preprocessor),
                ("selector", SelectKBest()),
                ("scaler", StandardScaler(with_mean=False)),
                (
                    "model",
                    LogisticRegression(
                        max_iter=1000,
                        class_weight="balanced",
                        random_state=42,
                    ),
                ),
            ]
        ),
        "XGBoost": Pipeline(
            [
                ("fe", FeatureEngineerTransformer(**round4_fe_params)),
                ("geo", GeoTransformer(strategy="drop")),
                ("prep", preprocessor),
                ("selector", SelectKBest()),
                (
                    "model",
                    XGBClassifier(
                        objective="binary:logistic",
                        eval_metric="logloss",
                        scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
                        random_state=42,
                        n_jobs=-1,
                    ),
                ),
            ]
        ),
        "MLP": Pipeline(
            [
                ("fe", FeatureEngineerTransformer(**round4_fe_params)),
                ("geo", GeoTransformer(strategy="drop")),
                ("prep", preprocessor),
                ("selector", SelectKBest()),
                ("scaler", StandardScaler(with_mean=False)),
                (
                    "model",
                    MLPClassifierWrapper(
                        hidden_dim=64,
                        batch_size=64,
                        lr=1e-3,
                        weight_decay=1e-5,
                        dropout=0.0,
                        max_epochs=80,
                        patience=16,
                        min_delta=1e-3,
                        val_size=0.15,
                        threshold=0.5,
                        random_state=42,
                        verbose=False,
                    ),
                ),
            ]
        ),
    }


round4_rows = []
feature_selection_searches = {}
selected_feature_logs = {}
round4_fold_results = {}

for model_name, estimator in build_round4_estimators().items():
    round4_logger.info("Round 4 | iniciando avaliacao do modelo=%s", model_name)

    for score_func, selector_name in selector_specs:
        for k_value in k_grid:
            round4_logger.info(
                "Round 4 | model=%s | selector=%s | k=%s",
                model_name,
                selector_name,
                k_value,
            )

            estimator_iter = clone(estimator)
            estimator_iter.set_params(
                selector__score_func=score_func,
                selector__k=k_value,
            )

            cv_res = cross_validate(
                estimator=estimator_iter,
                X=X_train_val,
                y=y_train_val,
                cv=cv,
                scoring=scoring,
                n_jobs=1,
                return_train_score=False,
            )

            experiment_name = f"{model_name}__{selector_name}__k_{k_value}"
            fold_df = pd.DataFrame(
                {
                    "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
                    "pr_auc": cv_res["test_pr_auc"],
                    "roc_auc": cv_res["test_roc_auc"],
                    "recall": cv_res["test_recall"],
                    "precision": cv_res["test_precision"],
                    "f1": cv_res["test_f1"],
                    "fit_time_s": cv_res["fit_time"],
                    "score_time_s": cv_res["score_time"],
                }
            )
            round4_fold_results[experiment_name] = fold_df

            fitted_full = clone(estimator_iter).fit(X_train_val, y_train_val)
            feature_selection_searches[experiment_name] = fitted_full
            selected_features = extract_selected_feature_names(
                fitted_full,
                processed_feature_names,
                selector_step="selector",
            )
            selected_feature_logs[experiment_name] = selected_features

            round4_rows.append(
                {
                    "model": model_name,
                    "selector": selector_name,
                    "k": int(k_value),
                    "pr_auc_mean": cv_res["test_pr_auc"].mean(),
                    "roc_auc_mean": cv_res["test_roc_auc"].mean(),
                    "recall_mean": cv_res["test_recall"].mean(),
                    "precision_mean": cv_res["test_precision"].mean(),
                    "f1_mean": cv_res["test_f1"].mean(),
                    "fit_time_mean_s": cv_res["fit_time"].mean(),
                    "score_time_mean_s": cv_res["score_time"].mean(),
                }
            )

results_fs = rank_round4_results(pd.DataFrame(round4_rows))

round4_results_by_model = {
    model_name: rank_round4_results(results_fs[results_fs["model"] == model_name].copy())
    for model_name in results_fs["model"].unique()
}

round4_k_candidates = {}
for model_name, df_model in round4_results_by_model.items():
    top_df = df_model.head(min(5, len(df_model))).copy()
    round4_k_candidates[model_name] = sorted(top_df["k"].astype(int).unique().tolist())
    round4_logger.info(
        "Round 4 | model=%s | faixa promissora de k=%s",
        model_name,
        round4_k_candidates[model_name],
    )

assert len(round4_results_by_model) == 3, "A rodada de feature selection deve retornar exatamente 3 modelos."

print("=== RESULTADOS FEATURE SELECTION (CONSOLIDADO) ===")
display(results_fs.round(4))

for model_name, df_model in round4_results_by_model.items():
    print(f"=== TOP CONFIGURACOES ROUND 4: {model_name} ===")
    display(df_model.head(10).round(4))

2026-04-30 13:46:00 [INFO] round4_feature_selection: Round 4 | total de features processadas=58 | k_grid=[10, 20, 30, 40, 50, 58]
2026-04-30 13:46:00 [INFO] round4_feature_selection: Round 4 | iniciando avaliacao do modelo=LogisticRegression
2026-04-30 13:46:00 [INFO] round4_feature_selection: Round 4 | model=LogisticRegression | selector=f_classif | k=10
2026-04-30 13:46:00 [INFO] round4_feature_selection: Round 4 | model=LogisticRegression | selector=f_classif | k=20
2026-04-30 13:46:00 [INFO] round4_feature_selection: Round 4 | model=LogisticRegression | selector=f_classif | k=30
2026-04-30 13:46:01 [INFO] round4_feature_selection: Round 4 | model=LogisticRegression | selector=f_classif | k=40
2026-04-30 13:46:02 [INFO] round4_feature_selection: Round 4 | model=LogisticRegression | selector=f_classif | k=50
2026-04-30 13:46:02 [INFO] round4_feature_selection: Round 4 | model=LogisticRegression | selector=f_classif | k=58
2026-04-30 13:46:03 [INFO] round4_feature_selection: Round 4 |

,model,selector,k,n_selected_features,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,XGBoost,f_classif,30,30,0.9529,0.9812,0.8799,0.8434,0.8611,0.0686,0.0283
1,XGBoost,f_classif,20,20,0.9526,0.9808,0.8953,0.8457,0.8695,0.0623,0.0277
2,XGBoost,mutual_info_classif,40,40,0.9525,0.9810,0.8792,0.8456,0.8619,0.4784,0.0284
3,XGBoost,mutual_info_classif,20,20,0.9520,0.9810,0.8807,0.8496,0.8647,0.5268,0.0291
4,XGBoost,f_classif,50,50,0.9519,0.9806,0.8784,0.8449,0.8613,0.0752,0.0291
5,XGBoost,mutual_info_classif,30,30,0.9513,0.9803,0.8792,0.8405,0.8593,0.4941,0.0282
6,XGBoost,f_classif,58,58,0.9512,0.9802,0.8738,0.8404,0.8566,0.0779,0.0288
7,XGBoost,mutual_info_classif,58,58,0.9512,0.9802,0.8738,0.8404,0.8566,0.5008,0.0306
8,XGBoost,mutual_info_classif,50,50,0.9510,0.9804,0.8754,0.8445,0.8595,0.4963,0.0297
9,XGBoost,f_classif,40,40,0.9509,0.9803,0.8792,0.8463,0.8623,0.0772,0.0284


=== TOP CONFIGURACOES ROUND 4: XGBoost ===


,model,selector,k,n_selected_features,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,XGBoost,f_classif,30,30,0.9529,0.9812,0.8799,0.8434,0.8611,0.0686,0.0283
1,XGBoost,f_classif,20,20,0.9526,0.9808,0.8953,0.8457,0.8695,0.0623,0.0277
2,XGBoost,mutual_info_classif,40,40,0.9525,0.9810,0.8792,0.8456,0.8619,0.4784,0.0284
3,XGBoost,mutual_info_classif,20,20,0.9520,0.9810,0.8807,0.8496,0.8647,0.5268,0.0291
4,XGBoost,f_classif,50,50,0.9519,0.9806,0.8784,0.8449,0.8613,0.0752,0.0291
5,XGBoost,mutual_info_classif,30,30,0.9513,0.9803,0.8792,0.8405,0.8593,0.4941,0.0282
6,XGBoost,f_classif,58,58,0.9512,0.9802,0.8738,0.8404,0.8566,0.0779,0.0288
7,XGBoost,mutual_info_classif,58,58,0.9512,0.9802,0.8738,0.8404,0.8566,0.5008,0.0306
8,XGBoost,mutual_info_classif,50,50,0.9510,0.9804,0.8754,0.8445,0.8595,0.4963,0.0297
9,XGBoost,f_classif,40,40,0.9509,0.9803,0.8792,0.8463,0.8623,0.0772,0.0284


=== TOP CONFIGURACOES ROUND 4: LogisticRegression ===


,model,selector,k,n_selected_features,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,LogisticRegression,f_classif,30,30,0.9398,0.9763,0.9342,0.7805,0.8502,0.0613,0.0221
1,LogisticRegression,f_classif,40,40,0.9398,0.9762,0.9274,0.7838,0.8493,0.0749,0.0237
2,LogisticRegression,f_classif,50,50,0.9397,0.9760,0.9289,0.7832,0.8496,0.0714,0.0236
3,LogisticRegression,f_classif,20,20,0.9395,0.9762,0.9312,0.7805,0.8490,0.0483,0.0227
4,LogisticRegression,mutual_info_classif,40,40,0.9394,0.9760,0.9304,0.7839,0.8507,0.5309,0.0259
5,LogisticRegression,f_classif,58,58,0.9393,0.9759,0.9297,0.7827,0.8497,0.0540,0.0244
6,LogisticRegression,mutual_info_classif,58,58,0.9393,0.9759,0.9297,0.7827,0.8497,0.5015,0.0267
7,LogisticRegression,mutual_info_classif,50,50,0.9392,0.9758,0.9266,0.7807,0.8472,0.5189,0.0242
8,LogisticRegression,mutual_info_classif,30,30,0.9391,0.9761,0.9312,0.7795,0.8483,0.4946,0.0233
9,LogisticRegression,mutual_info_classif,20,20,0.9379,0.9756,0.9319,0.7761,0.8467,0.4837,0.0220


=== TOP CONFIGURACOES ROUND 4: MLP ===


,model,selector,k,n_selected_features,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,MLP,f_classif,20,20,0.9387,0.9758,0.9358,0.7726,0.8458,1.6220,0.0240
1,MLP,mutual_info_classif,40,40,0.9384,0.9755,0.9312,0.7685,0.8414,1.9829,0.0243
2,MLP,mutual_info_classif,50,50,0.9376,0.9752,0.9319,0.7688,0.8416,1.7763,0.0255
3,MLP,mutual_info_classif,30,30,0.9374,0.9753,0.9304,0.7620,0.8372,2.0617,0.0244
4,MLP,f_classif,40,40,0.9372,0.9751,0.9289,0.7703,0.8414,1.9027,0.0261
5,MLP,mutual_info_classif,20,20,0.9365,0.9752,0.9358,0.7625,0.8394,2.2472,0.0240
6,MLP,f_classif,30,30,0.9365,0.9748,0.9358,0.7670,0.8423,1.7645,0.0236
7,MLP,f_classif,58,58,0.9363,0.9749,0.9289,0.7724,0.8426,1.5514,0.0247
8,MLP,f_classif,50,50,0.9357,0.9744,0.9243,0.7801,0.8443,1.6987,0.0264
9,MLP,mutual_info_classif,58,58,0.9355,0.9744,0.9327,0.7618,0.8384,1.9313,0.0261


#### L1-Based Selection - Regressão Logística

**Regras**

- Limitar a regularização para não passar de 10 features (mínimo exigido pelo projeto) → Early Stopping

In [24]:
import logging
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from src.features.feature_engineer_transformer import FeatureEngineerTransformer
from src.features.geo_transformer import GeoTransformer


# ------------------------------------------------------------
# Logging
# ------------------------------------------------------------
logger = logging.getLogger("round4_l1_logreg")
logger.setLevel(logging.INFO)
logger.propagate = False

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%H:%M:%S",
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def build_l1_logreg_pipeline(C, fe_params):
    return Pipeline([
        ("fe", FeatureEngineerTransformer(**fe_params)),
        ("geo", GeoTransformer(strategy="drop")),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            solver="saga",
            l1_ratio=1.0,
            C=C,
            max_iter=5000,
            class_weight="balanced",
            random_state=42,
        )),
    ])


def get_nonzero_feature_names(fitted_pipeline):
    feature_names = fitted_pipeline.named_steps["prep"].get_feature_names_out()
    coefs = fitted_pipeline.named_steps["model"].coef_.ravel()
    nonzero_mask = coefs != 0
    selected_features = np.asarray(feature_names)[nonzero_mask].tolist()
    return selected_features, coefs


# ------------------------------------------------------------
# Grid de C
# Mais a  esquerda = regularização mais fraca
# Mais a  direita = regularização mais forte
# A busca para quando < 10 features restarem
# ------------------------------------------------------------
c_grid = [
    10.0, 7.5, 5.0, 3.0, 2.0, 1.0,
    0.75, 0.5, 0.3, 0.2, 0.1,
    0.075, 0.05, 0.04, 0.03, 0.02, 0.01,
    0.0075, 0.005, 0.004, 0.003, 0.002, 0.001,
]

min_features_threshold = 10
metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]

rows = []
l1_search_logs = {}
l1_search_fold_results = {}

for C in c_grid:
    logger.info("Iniciando avaliação com L1 LogisticRegression | C=%.5f", C)

    estimator = build_l1_logreg_pipeline(
        C=C,
        fe_params=round4_fe_params,
    )

    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
        return_estimator=True,
    )

    fold_df = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{metric: cv_res[f"test_{metric}"] for metric in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    l1_search_fold_results[C] = fold_df

    # Refit no conjunto completo para extrair as features finais selecionadas
    fitted_full = clone(estimator).fit(X_train_val, y_train_val)
    selected_features, coefs = get_nonzero_feature_names(fitted_full)
    n_selected = len(selected_features)

    row = {
        "model": "LogisticRegression_L1",
        "C": C,
        "n_selected_features": n_selected,
        "pr_auc_mean": cv_res["test_pr_auc"].mean(),
        "roc_auc_mean": cv_res["test_roc_auc"].mean(),
        "recall_mean": cv_res["test_recall"].mean(),
        "precision_mean": cv_res["test_precision"].mean(),
        "f1_mean": cv_res["test_f1"].mean(),
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    }
    rows.append(row)

    l1_search_logs[C] = {
        "selected_features": selected_features,
        "coefficients": coefs,
        "metrics": row,
    }

    logger.info(
        (
            "Resultado | C=%.5f | selected=%d | "
            "PR-AUC=%.4f | ROC-AUC=%.4f | Recall=%.4f | Precision=%.4f | F1=%.4f"
        ),
        C,
        n_selected,
        row["pr_auc_mean"],
        row["roc_auc_mean"],
        row["recall_mean"],
        row["precision_mean"],
        row["f1_mean"],
    )

    logger.info("Features selecionadas (%d): %s", n_selected, selected_features)

    if n_selected < min_features_threshold:
        logger.info(
            (
                "Early stopping acionado: número de features selecionadas "
                "(%d) ficou abaixo do limite de %d."
            ),
            n_selected,
            min_features_threshold,
        )
        break

results_l1_fs = (
    pd.DataFrame(rows)
    .sort_values(["pr_auc_mean", "roc_auc_mean", "recall_mean"], ascending=[False, False, False])
    .reset_index(drop=True)
)

display(results_l1_fs.round(4))


13:49:17 | INFO | Iniciando avaliação com L1 LogisticRegression | C=10.00000
13:49:25 | INFO | Resultado | C=10.00000 | selected=47 | PR-AUC=0.9388 | ROC-AUC=0.9758 | Recall=0.9289 | Precision=0.7821 | F1=0.8490
13:49:25 | INFO | Features selecionadas (47): ['cat__Gender_Female', 'cat__Senior Citizen_No', 'cat__Partner_Yes', 'cat__Dependents_Yes', 'cat__Phone Service_No', 'cat__Phone Service_Yes', 'cat__Multiple Lines_No', 'cat__Multiple Lines_No phone service', 'cat__Internet Service_DSL', 'cat__Internet Service_Fiber optic', 'cat__Internet Service_No', 'cat__Online Security_No internet service', 'cat__Online Security_Yes', 'cat__Online Backup_No internet service', 'cat__Online Backup_Yes', 'cat__Device Protection_No', 'cat__Device Protection_No internet service', 'cat__Device Protection_Yes', 'cat__Tech Support_No', 'cat__Tech Support_No internet service', 'cat__Tech Support_Yes', 'cat__Streaming TV_No', 'cat__Streaming TV_No internet service', 'cat__Streaming TV_Yes', 'cat__Streamin

,model,C,n_selected_features,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,LogisticRegression_L1,0.0750,28,0.9400,0.9762,0.9327,0.7782,0.8483,0.4984,0.0206
1,LogisticRegression_L1,0.1000,28,0.9400,0.9762,0.9289,0.7786,0.8469,0.6142,0.0207
2,LogisticRegression_L1,0.0400,26,0.9399,0.9762,0.9350,0.7717,0.8454,0.3550,0.0202
3,LogisticRegression_L1,0.2000,29,0.9399,0.9761,0.9281,0.7819,0.8486,1.1292,0.0207
4,LogisticRegression_L1,0.0500,26,0.9399,0.9762,0.9335,0.7730,0.8455,0.3913,0.0202
5,LogisticRegression_L1,0.3000,34,0.9397,0.9760,0.9274,0.7827,0.8488,1.4610,0.0211
6,LogisticRegression_L1,0.0300,24,0.9396,0.9762,0.9358,0.7690,0.8440,0.3345,0.0212
7,LogisticRegression_L1,0.5000,35,0.9396,0.9759,0.9258,0.7815,0.8474,1.5760,0.0211
8,LogisticRegression_L1,0.7500,34,0.9394,0.9759,0.9266,0.7832,0.8487,2.1551,0.0212
9,LogisticRegression_L1,1.0000,36,0.9393,0.9758,0.9274,0.7828,0.8488,2.5959,0.0212


In [35]:
thr = results_l1_fs['pr_auc_mean'].quantile(0.80)
top_trials_l1 = results_l1_fs[results_l1_fs['pr_auc_mean'] >= thr]
top_trials_l1 = top_trials_l1.sort_values('pr_auc_mean', ascending=False).reset_index(drop=True)
top_trials_l1.describe()

,C,n_selected_features,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
count,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000
mean,0.093000,27.400000,0.939954,0.976187,0.931644,0.776674,0.846946,0.597625,0.020475
std,0.064187,1.341641,0.000044,0.000050,0.002992,0.004214,0.001495,0.313919,0.000295
min,0.040000,26.000000,0.939915,0.976107,0.928124,0.771739,0.845402,0.354970,0.020153
25%,0.050000,26.000000,0.939920,0.976181,0.928891,0.772975,0.845527,0.391329,0.020158
50%,0.075000,28.000000,0.939936,0.976196,0.932716,0.778221,0.846907,0.498389,0.020603
75%,0.100000,28.000000,0.939983,0.976208,0.933480,0.778557,0.848318,0.614190,0.020723
max,0.200000,29.000000,0.940017,0.976241,0.935009,0.781876,0.848576,1.129245,0.020735


### Conclusão

## Fine Tuning de Hiperparametros em 2 Etapas

**Objetivo:**
- fazer uma exploracao inicial com `RandomizedSearchCV` otimizando `PR-AUC`;
- gerar um dataset de resultados para a `MLP` e outro para o `XGBoost`;
- usar os melhores trials para definir uma faixa reduzida de busca na segunda etapa com `Optuna`.

### Etapa 1 - RandomizedSearchCV

#### MLP

In [25]:
mlp_logger = get_logger("round4_mlp_random_search")

mlp_selector_candidates = round4_k_candidates.get("MLP") or k_grid
mlp_param_distributions = {
    "selector__score_func": [f_classif, mutual_info_classif],
    "selector__k": mlp_selector_candidates,
    "model__activation": ["relu", "leaky_relu", "elu", "gelu", "tanh"],
    "model__hidden_dim": [32, 64, 128],
    "model__dropout": [0.0, 0.1, 0.2, 0.3],
    "model__lr": [1e-4, 3e-4, 1e-3, 3e-3],
    "model__weight_decay": [0.0, 1e-6, 1e-5, 1e-4],
    "model__batch_size": [32, 64, 128],
}

mlp_base_pipeline = Pipeline(
    [
        ("fe", FeatureEngineerTransformer(**round4_fe_params)),
        ("geo", GeoTransformer(strategy="drop")),
        ("prep", preprocessor),
        ("selector", SelectKBest()),
        ("scaler", StandardScaler(with_mean=False)),
        (
            "model",
            MLPClassifierWrapper(
                output_dim=1,
                max_epochs=80,
                patience=16,
                min_delta=1e-3,
                threshold=0.5,
                val_size=0.15,
                random_state=42,
                verbose=False,
            ),
        ),
    ]
)

mlp_random_search = RandomizedSearchCV(
    estimator=mlp_base_pipeline,
    param_distributions=mlp_param_distributions,
    n_iter=100,
    scoring=scoring,
    refit="pr_auc",
    cv=cv,
    n_jobs=1,
    random_state=42,
    return_train_score=False,
    verbose=1,
)
mlp_random_search.fit(X_train_val, y_train_val)

mlp_rows = []
mlp_cv_results = mlp_random_search.cv_results_
for trial_idx, params in enumerate(mlp_cv_results["params"], start=1):
    selector_func = params["selector__score_func"]
    mlp_rows.append(
        {
            "trial_id": trial_idx,
            "stage": "random_search",
            "model": "MLP",
            "selector": selector_label_map.get(selector_func, getattr(selector_func, "__name__", str(selector_func))),
            "k": int(params["selector__k"]),
            "activation": params["model__activation"],
            "hidden_dim": int(params["model__hidden_dim"]),
            "dropout": float(params["model__dropout"]),
            "lr": float(params["model__lr"]),
            "weight_decay": float(params["model__weight_decay"]),
            "batch_size": int(params["model__batch_size"]),
            "max_epochs": 80,
            "patience": 16,
            "threshold": 0.5,
            "pr_auc_mean": mlp_cv_results["mean_test_pr_auc"][trial_idx - 1],
            "roc_auc_mean": mlp_cv_results["mean_test_roc_auc"][trial_idx - 1],
            "recall_mean": mlp_cv_results["mean_test_recall"][trial_idx - 1],
            "precision_mean": mlp_cv_results["mean_test_precision"][trial_idx - 1],
            "f1_mean": mlp_cv_results["mean_test_f1"][trial_idx - 1],
            "fit_time_mean_s": mlp_cv_results["mean_fit_time"][trial_idx - 1],
            "score_time_mean_s": mlp_cv_results["mean_score_time"][trial_idx - 1],
        }
    )

results_mlp_random_search = (
    pd.DataFrame(mlp_rows)
    .sort_values(["pr_auc_mean", "roc_auc_mean", "recall_mean"], ascending=[False, False, False])
    .reset_index(drop=True)
)
results_mlp_grid = results_mlp_random_search.copy()

best_mlp_estimator = mlp_random_search.best_estimator_
best_mlp_params = mlp_random_search.best_params_
best_mlp_selected_features = extract_selected_feature_names(
    best_mlp_estimator,
    processed_feature_names,
    selector_step="selector",
)
best_mlp_result = results_mlp_random_search.iloc[0].to_dict()

mlp_logger.info("RandomizedSearchCV da MLP finalizado | top PR-AUC=%.4f", best_mlp_result["pr_auc_mean"])
mlp_logger.info("Melhores parametros da MLP: %s", best_mlp_params)
mlp_logger.info("Features selecionadas na melhor configuracao da MLP: %s", best_mlp_selected_features)

display(results_mlp_random_search.head(20).round(4))

Fitting 5 folds for each of 100 candidates, totalling 500 fits
2026-04-30 14:16:27 [INFO] round4_mlp_random_search: RandomizedSearchCV da MLP finalizado | top PR-AUC=0.9393
2026-04-30 14:16:27 [INFO] round4_mlp_random_search: Melhores parametros da MLP: {'selector__score_func': <function f_classif at 0x000001D5F36117A0>, 'selector__k': 40, 'model__weight_decay': 1e-05, 'model__lr': 0.0003, 'model__hidden_dim': 32, 'model__dropout': 0.0, 'model__batch_size': 32, 'model__activation': 'elu'}
2026-04-30 14:16:27 [INFO] round4_mlp_random_search: Features selecionadas na melhor configuracao da MLP: ['cat__Senior Citizen_No', 'cat__Partner_No', 'cat__Partner_Yes', 'cat__Dependents_No', 'cat__Dependents_Yes', 'cat__Internet Service_Fiber optic', 'cat__Internet Service_No', 'cat__Online Security_No', 'cat__Online Security_No internet service', 'cat__Online Security_Yes', 'cat__Online Backup_No', 'cat__Online Backup_No internet service', 'cat__Device Protection_No', 'cat__Device Protection_No in

,trial_id,stage,model,selector,k,activation,hidden_dim,dropout,lr,weight_decay,...,max_epochs,patience,threshold,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,81,random_search,MLP,f_classif,40,elu,32,0.0,0.0003,0.0000,...,80,16,0.5,0.9393,0.9761,0.9335,0.7744,0.8459,4.6791,0.0264
1,69,random_search,MLP,f_classif,50,elu,128,0.1,0.0003,0.0001,...,80,16,0.5,0.9393,0.9759,0.9250,0.7770,0.8436,2.6914,0.0287
2,64,random_search,MLP,mutual_info_classif,40,elu,32,0.0,0.0003,0.0000,...,80,16,0.5,0.9391,0.9758,0.9281,0.7755,0.8446,3.5951,0.0284
3,84,random_search,MLP,mutual_info_classif,30,tanh,32,0.3,0.0030,0.0001,...,80,16,0.5,0.9389,0.9759,0.9342,0.7758,0.8469,1.9476,0.0241
4,7,random_search,MLP,f_classif,50,tanh,64,0.1,0.0030,0.0000,...,80,16,0.5,0.9389,0.9757,0.9273,0.7706,0.8413,0.9864,0.0245
5,97,random_search,MLP,f_classif,50,gelu,32,0.0,0.0003,0.0001,...,80,16,0.5,0.9388,0.9757,0.9266,0.7736,0.8426,2.8783,0.0228
6,55,random_search,MLP,f_classif,50,elu,128,0.3,0.0003,0.0000,...,80,16,0.5,0.9388,0.9757,0.9258,0.7804,0.8463,3.6022,0.0290
7,31,random_search,MLP,f_classif,50,tanh,128,0.1,0.0003,0.0000,...,80,16,0.5,0.9388,0.9755,0.9243,0.7684,0.8385,4.1043,0.0288
8,30,random_search,MLP,mutual_info_classif,40,relu,64,0.0,0.0003,0.0001,...,80,16,0.5,0.9387,0.9757,0.9342,0.7716,0.8447,4.7094,0.0286
9,78,random_search,MLP,mutual_info_classif,50,tanh,64,0.3,0.0003,0.0000,...,80,16,0.5,0.9386,0.9758,0.9304,0.7724,0.8436,2.9243,0.0284


In [26]:
assert best_mlp_result is not None, "Nenhum resultado foi encontrado na busca da MLP."
assert best_mlp_params is not None, "best_mlp_params nao foi definido."
assert best_mlp_estimator is not None, "best_mlp_estimator nao foi definido."

best_mlp_summary = pd.DataFrame([best_mlp_result]).round(4)
mlp_top_trials = results_mlp_random_search.head(min(20, len(results_mlp_random_search))).copy()

mlp_optuna_seed_space = {
    "selector_name": sorted(mlp_top_trials["selector"].dropna().unique().tolist()),
    "k_values": sorted(mlp_top_trials["k"].astype(int).unique().tolist()),
    "activation_values": sorted(mlp_top_trials["activation"].dropna().unique().tolist()),
    "hidden_dim_values": sorted(mlp_top_trials["hidden_dim"].astype(int).unique().tolist()),
    "batch_size_values": sorted(mlp_top_trials["batch_size"].astype(int).unique().tolist()),
    "dropout_range": (
        float(mlp_top_trials["dropout"].min()),
        float(mlp_top_trials["dropout"].max()),
    ),
    "lr_range": (
        float(mlp_top_trials["lr"].min()),
        float(mlp_top_trials["lr"].max()),
    ),
    "weight_decay_range": (
        float(mlp_top_trials["weight_decay"].min()),
        float(mlp_top_trials["weight_decay"].max()),
    ),
}

mlp_optuna_space_preview = pd.DataFrame(
    [
        {"parameter": "selector_name", "search_space": mlp_optuna_seed_space["selector_name"]},
        {"parameter": "k_values", "search_space": mlp_optuna_seed_space["k_values"]},
        {"parameter": "activation_values", "search_space": mlp_optuna_seed_space["activation_values"]},
        {"parameter": "hidden_dim_values", "search_space": mlp_optuna_seed_space["hidden_dim_values"]},
        {"parameter": "batch_size_values", "search_space": mlp_optuna_seed_space["batch_size_values"]},
        {"parameter": "dropout_range", "search_space": mlp_optuna_seed_space["dropout_range"]},
        {"parameter": "lr_range", "search_space": mlp_optuna_seed_space["lr_range"]},
        {"parameter": "weight_decay_range", "search_space": mlp_optuna_seed_space["weight_decay_range"]},
    ]
)

display(best_mlp_summary)
display(mlp_optuna_space_preview)

,trial_id,stage,model,selector,k,activation,hidden_dim,dropout,lr,weight_decay,...,max_epochs,patience,threshold,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,81,random_search,MLP,f_classif,40,elu,32,0.0,0.0003,0.0,...,80,16,0.5,0.9393,0.9761,0.9335,0.7744,0.8459,4.6791,0.0264


,parameter,search_space
0,selector_name,"[f_classif, mutual_info_classif]"
1,k_values,"[20, 30, 40, 50]"
2,activation_values,"[elu, gelu, leaky_relu, relu, tanh]"
3,hidden_dim_values,"[32, 64, 128]"
4,batch_size_values,"[32, 64, 128]"
5,dropout_range,"(0.0, 0.3)"
6,lr_range,"(0.0001, 0.003)"
7,weight_decay_range,"(0.0, 0.0001)"


In [29]:
mlp_selector_candidates

[20, 30, 40, 50]

#### XGBoost

In [27]:
xgb_logger = get_logger("round4_xgb_random_search")

xgb_param_distributions = {
    "model__n_estimators": [200, 300, 500, 700],
    "model__max_depth": [3, 5, 7, 9],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__min_child_weight": [1, 3, 5, 7],
    "model__subsample": [0.7, 0.85, 1.0],
    "model__colsample_bytree": [0.7, 0.85, 1.0],
    "model__gamma": [0.0, 0.1, 0.3, 0.5],
    "model__reg_alpha": [0.0, 0.1, 0.5, 1.0],
    "model__reg_lambda": [1.0, 2.0, 5.0, 10.0],
}

xgb_base_pipeline = Pipeline(
    [
        ("fe", FeatureEngineerTransformer(**round4_fe_params)),
        ("geo", GeoTransformer(strategy="drop")),
        ("prep", preprocessor),
        (
            "model",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

xgb_random_search = RandomizedSearchCV(
    estimator=xgb_base_pipeline,
    param_distributions=xgb_param_distributions,
    n_iter=100,
    scoring=scoring,
    refit="pr_auc",
    cv=cv,
    n_jobs=1,
    random_state=42,
    return_train_score=False,
    verbose=1,
)
xgb_random_search.fit(X_train_val, y_train_val)

xgb_rows = []
xgb_cv_results = xgb_random_search.cv_results_
for trial_idx, params in enumerate(xgb_cv_results["params"], start=1):
    xgb_rows.append(
        {
            "trial_id": trial_idx,
            "stage": "random_search",
            "model": "XGBoost",
            "n_estimators": int(params["model__n_estimators"]),
            "max_depth": int(params["model__max_depth"]),
            "learning_rate": float(params["model__learning_rate"]),
            "min_child_weight": float(params["model__min_child_weight"]),
            "subsample": float(params["model__subsample"]),
            "colsample_bytree": float(params["model__colsample_bytree"]),
            "gamma": float(params["model__gamma"]),
            "reg_alpha": float(params["model__reg_alpha"]),
            "reg_lambda": float(params["model__reg_lambda"]),
            "pr_auc_mean": xgb_cv_results["mean_test_pr_auc"][trial_idx - 1],
            "roc_auc_mean": xgb_cv_results["mean_test_roc_auc"][trial_idx - 1],
            "recall_mean": xgb_cv_results["mean_test_recall"][trial_idx - 1],
            "precision_mean": xgb_cv_results["mean_test_precision"][trial_idx - 1],
            "f1_mean": xgb_cv_results["mean_test_f1"][trial_idx - 1],
            "fit_time_mean_s": xgb_cv_results["mean_fit_time"][trial_idx - 1],
            "score_time_mean_s": xgb_cv_results["mean_score_time"][trial_idx - 1],
        }
    )

results_xgb_random_search = (
    pd.DataFrame(xgb_rows)
    .sort_values(["pr_auc_mean", "roc_auc_mean", "recall_mean"], ascending=[False, False, False])
    .reset_index(drop=True)
)
results_xgb_grid = results_xgb_random_search.copy()

best_xgb_estimator = xgb_random_search.best_estimator_
best_xgb_params = xgb_random_search.best_params_
best_xgb_result = results_xgb_random_search.iloc[0].to_dict()

xgb_logger.info("RandomizedSearchCV do XGBoost finalizado | top PR-AUC=%.4f", best_xgb_result["pr_auc_mean"])
xgb_logger.info("Melhores parametros do XGBoost: %s", best_xgb_params)

display(results_xgb_random_search.head(20).round(4))

Fitting 5 folds for each of 100 candidates, totalling 500 fits
2026-04-30 14:18:01 [INFO] round4_xgb_random_search: RandomizedSearchCV do XGBoost finalizado | top PR-AUC=0.9603
2026-04-30 14:18:01 [INFO] round4_xgb_random_search: Melhores parametros do XGBoost: {'model__subsample': 1.0, 'model__reg_lambda': 5.0, 'model__reg_alpha': 0.1, 'model__n_estimators': 500, 'model__min_child_weight': 3, 'model__max_depth': 3, 'model__learning_rate': 0.03, 'model__gamma': 0.1, 'model__colsample_bytree': 0.85}


,trial_id,stage,model,n_estimators,max_depth,learning_rate,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,15,random_search,XGBoost,500,3,0.03,3.0,1.00,0.85,0.1,0.1,5.0,0.9603,0.9843,0.9472,0.8073,0.8715,0.1223,0.0268
1,61,random_search,XGBoost,300,3,0.03,5.0,1.00,0.70,0.5,0.5,2.0,0.9600,0.9843,0.9503,0.8042,0.8710,0.0896,0.0270
2,52,random_search,XGBoost,200,3,0.05,3.0,0.85,0.85,0.3,0.1,1.0,0.9599,0.9842,0.9480,0.8058,0.8710,0.0714,0.0265
3,91,random_search,XGBoost,200,3,0.05,1.0,0.85,1.00,0.0,1.0,2.0,0.9597,0.9841,0.9457,0.8050,0.8695,0.0755,0.0270
4,39,random_search,XGBoost,300,5,0.03,1.0,1.00,0.70,0.0,0.0,1.0,0.9596,0.9841,0.9427,0.8107,0.8716,0.1073,0.0268
5,47,random_search,XGBoost,500,5,0.01,7.0,1.00,1.00,0.3,1.0,1.0,0.9596,0.9841,0.9518,0.8017,0.8702,0.1561,0.0271
6,42,random_search,XGBoost,200,3,0.05,5.0,0.70,1.00,0.5,0.0,2.0,0.9595,0.9841,0.9457,0.8114,0.8732,0.0722,0.0266
7,12,random_search,XGBoost,300,3,0.03,7.0,0.70,0.70,0.1,1.0,2.0,0.9595,0.9841,0.9480,0.8079,0.8722,0.0917,0.0265
8,13,random_search,XGBoost,200,5,0.03,5.0,1.00,0.70,0.5,0.0,1.0,0.9595,0.9840,0.9472,0.8062,0.8709,0.0856,0.0270
9,41,random_search,XGBoost,700,5,0.01,5.0,0.85,0.85,0.5,0.0,1.0,0.9594,0.9840,0.9449,0.8117,0.8731,0.2259,0.0277


In [28]:
assert best_xgb_result is not None, "Nenhum resultado foi encontrado na busca do XGBoost."
assert best_xgb_params is not None, "best_xgb_params nao foi definido."
assert best_xgb_estimator is not None, "best_xgb_estimator nao foi definido."

best_xgb_summary = pd.DataFrame([best_xgb_result]).round(4)
xgb_top_trials = results_xgb_random_search.head(min(20, len(results_xgb_random_search))).copy()

xgb_optuna_seed_space = {
    "n_estimators_values": sorted(xgb_top_trials["n_estimators"].astype(int).unique().tolist()),
    "max_depth_values": sorted(xgb_top_trials["max_depth"].astype(int).unique().tolist()),
    "min_child_weight_values": sorted(xgb_top_trials["min_child_weight"].astype(int).unique().tolist()),
    "learning_rate_range": (
        float(xgb_top_trials["learning_rate"].min()),
        float(xgb_top_trials["learning_rate"].max()),
    ),
    "subsample_range": (
        float(xgb_top_trials["subsample"].min()),
        float(xgb_top_trials["subsample"].max()),
    ),
    "colsample_bytree_range": (
        float(xgb_top_trials["colsample_bytree"].min()),
        float(xgb_top_trials["colsample_bytree"].max()),
    ),
    "gamma_range": (
        float(xgb_top_trials["gamma"].min()),
        float(xgb_top_trials["gamma"].max()),
    ),
    "reg_alpha_range": (
        float(xgb_top_trials["reg_alpha"].min()),
        float(xgb_top_trials["reg_alpha"].max()),
    ),
    "reg_lambda_range": (
        float(xgb_top_trials["reg_lambda"].min()),
        float(xgb_top_trials["reg_lambda"].max()),
    ),
}

xgb_optuna_space_preview = pd.DataFrame(
    [
        {"parameter": "n_estimators_values", "search_space": xgb_optuna_seed_space["n_estimators_values"]},
        {"parameter": "max_depth_values", "search_space": xgb_optuna_seed_space["max_depth_values"]},
        {"parameter": "min_child_weight_values", "search_space": xgb_optuna_seed_space["min_child_weight_values"]},
        {"parameter": "learning_rate_range", "search_space": xgb_optuna_seed_space["learning_rate_range"]},
        {"parameter": "subsample_range", "search_space": xgb_optuna_seed_space["subsample_range"]},
        {"parameter": "colsample_bytree_range", "search_space": xgb_optuna_seed_space["colsample_bytree_range"]},
        {"parameter": "gamma_range", "search_space": xgb_optuna_seed_space["gamma_range"]},
        {"parameter": "reg_alpha_range", "search_space": xgb_optuna_seed_space["reg_alpha_range"]},
        {"parameter": "reg_lambda_range", "search_space": xgb_optuna_seed_space["reg_lambda_range"]},
    ]
)

display(best_xgb_summary)
display(xgb_optuna_space_preview)

,trial_id,stage,model,n_estimators,max_depth,learning_rate,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,15,random_search,XGBoost,500,3,0.03,3.0,1.0,0.85,0.1,0.1,5.0,0.9603,0.9843,0.9472,0.8073,0.8715,0.1223,0.0268


,parameter,search_space
0,n_estimators_values,"[200, 300, 500, 700]"
1,max_depth_values,"[3, 5, 7]"
2,min_child_weight_values,"[1, 3, 5, 7]"
3,learning_rate_range,"(0.01, 0.1)"
4,subsample_range,"(0.7, 1.0)"
5,colsample_bytree_range,"(0.7, 1.0)"
6,gamma_range,"(0.0, 0.5)"
7,reg_alpha_range,"(0.0, 1.0)"
8,reg_lambda_range,"(1.0, 10.0)"


### Etapa 2 - Optuna em Faixa Reduzida

In [ ]:
try:
    import optuna
except ImportError as exc:
    raise ImportError(
        "Optuna nao esta instalado no ambiente atual. Atualize o ambiente apos a inclusao em pyproject.toml antes de executar esta etapa."
    ) from exc

optuna.logging.set_verbosity(optuna.logging.WARNING)

MLP_OPTUNA_TRIALS = 50
XGB_OPTUNA_TRIALS = 50
optuna_sort_columns = ["pr_auc_mean", "roc_auc_mean", "recall_mean"]


def sort_experiment_frame(df):
    return df.sort_values(optuna_sort_columns, ascending=[False, False, False]).reset_index(drop=True)


def suggest_float_or_fixed(trial, name, low, high, *, log=False):
    low = float(low)
    high = float(high)
    if np.isclose(low, high):
        return low
    if log and low > 0 and high > 0:
        return trial.suggest_float(name, low, high, log=True)
    return trial.suggest_float(name, low, high)

### Etapa 2 - Execucao do Optuna para MLP e XGBoost

In [ ]:
selector_name_to_func = {
    "f_classif": f_classif,
    "mutual_info_classif": mutual_info_classif,
}

mlp_optuna_rows = []


def build_mlp_optuna_pipeline_from_trial(trial):
    selector_name = trial.suggest_categorical("selector_name", mlp_optuna_seed_space["selector_name"])
    selector_func = selector_name_to_func[selector_name]

    return Pipeline(
        [
            ("fe", FeatureEngineerTransformer(**round4_fe_params)),
            ("geo", GeoTransformer(strategy="drop")),
            ("prep", preprocessor),
            (
                "selector",
                SelectKBest(
                    score_func=selector_func,
                    k=trial.suggest_categorical("k", mlp_optuna_seed_space["k_values"]),
                ),
            ),
            ("scaler", StandardScaler(with_mean=False)),
            (
                "model",
                MLPClassifierWrapper(
                    activation=trial.suggest_categorical("activation", mlp_optuna_seed_space["activation_values"]),
                    hidden_dim=trial.suggest_categorical("hidden_dim", mlp_optuna_seed_space["hidden_dim_values"]),
                    batch_size=trial.suggest_categorical("batch_size", mlp_optuna_seed_space["batch_size_values"]),
                    dropout=suggest_float_or_fixed(
                        trial,
                        "dropout",
                        *mlp_optuna_seed_space["dropout_range"],
                        log=False,
                    ),
                    lr=suggest_float_or_fixed(
                        trial,
                        "lr",
                        *mlp_optuna_seed_space["lr_range"],
                        log=True,
                    ),
                    weight_decay=suggest_float_or_fixed(
                        trial,
                        "weight_decay",
                        *mlp_optuna_seed_space["weight_decay_range"],
                        log=False,
                    ),
                    output_dim=1,
                    max_epochs=80,
                    patience=16,
                    min_delta=1e-3,
                    threshold=0.5,
                    val_size=0.15,
                    random_state=42,
                    verbose=False,
                ),
            ),
        ]
    )


def mlp_objective(trial):
    estimator = build_mlp_optuna_pipeline_from_trial(trial)
    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    row = {
        "trial_id": trial.number + 1,
        "stage": "optuna",
        "model": "MLP",
        **trial.params,
        "pr_auc_mean": cv_res["test_pr_auc"].mean(),
        "roc_auc_mean": cv_res["test_roc_auc"].mean(),
        "recall_mean": cv_res["test_recall"].mean(),
        "precision_mean": cv_res["test_precision"].mean(),
        "f1_mean": cv_res["test_f1"].mean(),
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    }
    mlp_optuna_rows.append(row)
    return row["pr_auc_mean"]


mlp_study = optuna.create_study(direction="maximize", study_name="round4_mlp_optuna")
mlp_study.optimize(mlp_objective, n_trials=MLP_OPTUNA_TRIALS)

results_mlp_optuna = sort_experiment_frame(pd.DataFrame(mlp_optuna_rows))
best_mlp_optuna_params = mlp_study.best_params
best_mlp_optuna_estimator = build_mlp_optuna_pipeline_from_trial(
    optuna.trial.FixedTrial(best_mlp_optuna_params)
).fit(X_train_val, y_train_val)
best_mlp_optuna_result = results_mlp_optuna.iloc[0].to_dict()

display(results_mlp_optuna.head(20).round(4))

xgb_optuna_rows = []


def build_xgb_optuna_pipeline_from_trial(trial):
    return Pipeline(
        [
            ("fe", FeatureEngineerTransformer(**round4_fe_params)),
            ("geo", GeoTransformer(strategy="drop")),
            ("prep", preprocessor),
            (
                "model",
                XGBClassifier(
                    objective="binary:logistic",
                    eval_metric="logloss",
                    scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
                    random_state=42,
                    n_jobs=-1,
                    n_estimators=trial.suggest_categorical("n_estimators", xgb_optuna_seed_space["n_estimators_values"]),
                    max_depth=trial.suggest_categorical("max_depth", xgb_optuna_seed_space["max_depth_values"]),
                    min_child_weight=trial.suggest_categorical("min_child_weight", xgb_optuna_seed_space["min_child_weight_values"]),
                    learning_rate=suggest_float_or_fixed(
                        trial,
                        "learning_rate",
                        *xgb_optuna_seed_space["learning_rate_range"],
                        log=True,
                    ),
                    subsample=suggest_float_or_fixed(
                        trial,
                        "subsample",
                        *xgb_optuna_seed_space["subsample_range"],
                        log=False,
                    ),
                    colsample_bytree=suggest_float_or_fixed(
                        trial,
                        "colsample_bytree",
                        *xgb_optuna_seed_space["colsample_bytree_range"],
                        log=False,
                    ),
                    gamma=suggest_float_or_fixed(
                        trial,
                        "gamma",
                        *xgb_optuna_seed_space["gamma_range"],
                        log=False,
                    ),
                    reg_alpha=suggest_float_or_fixed(
                        trial,
                        "reg_alpha",
                        *xgb_optuna_seed_space["reg_alpha_range"],
                        log=False,
                    ),
                    reg_lambda=suggest_float_or_fixed(
                        trial,
                        "reg_lambda",
                        *xgb_optuna_seed_space["reg_lambda_range"],
                        log=True,
                    ),
                ),
            ),
        ]
    )


def xgb_objective(trial):
    estimator = build_xgb_optuna_pipeline_from_trial(trial)
    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    row = {
        "trial_id": trial.number + 1,
        "stage": "optuna",
        "model": "XGBoost",
        **trial.params,
        "pr_auc_mean": cv_res["test_pr_auc"].mean(),
        "roc_auc_mean": cv_res["test_roc_auc"].mean(),
        "recall_mean": cv_res["test_recall"].mean(),
        "precision_mean": cv_res["test_precision"].mean(),
        "f1_mean": cv_res["test_f1"].mean(),
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    }
    xgb_optuna_rows.append(row)
    return row["pr_auc_mean"]


xgb_study = optuna.create_study(direction="maximize", study_name="round4_xgb_optuna")
xgb_study.optimize(xgb_objective, n_trials=XGB_OPTUNA_TRIALS)

results_xgb_optuna = sort_experiment_frame(pd.DataFrame(xgb_optuna_rows))
best_xgb_optuna_params = xgb_study.best_params
best_xgb_optuna_estimator = build_xgb_optuna_pipeline_from_trial(
    optuna.trial.FixedTrial(best_xgb_optuna_params)
).fit(X_train_val, y_train_val)
best_xgb_optuna_result = results_xgb_optuna.iloc[0].to_dict()

display(results_xgb_optuna.head(20).round(4))

### Etapa 2 - Optuna em Faixa Reduzida

In [ ]:
try:
    import optuna
except ImportError as exc:
    raise ImportError(
        "Optuna nao esta instalado no ambiente atual. Atualize o ambiente apos a inclusao em pyproject.toml antes de executar esta etapa."
    ) from exc

optuna.logging.set_verbosity(optuna.logging.WARNING)

MLP_OPTUNA_TRIALS = 50
XGB_OPTUNA_TRIALS = 50
optuna_sort_columns = ["pr_auc_mean", "roc_auc_mean", "recall_mean"]


def sort_experiment_frame(df):
    return df.sort_values(optuna_sort_columns, ascending=[False, False, False]).reset_index(drop=True)


def suggest_float_or_fixed(trial, name, low, high, *, log=False):
    low = float(low)
    high = float(high)
    if np.isclose(low, high):
        return low
    if log and low > 0 and high > 0:
        return trial.suggest_float(name, low, high, log=True)
    return trial.suggest_float(name, low, high)

### Etapa 2 - Execucao do Optuna para MLP e XGBoost

In [ ]:
selector_name_to_func = {
    "f_classif": f_classif,
    "mutual_info_classif": mutual_info_classif,
}

mlp_optuna_rows = []


def build_mlp_optuna_pipeline_from_trial(trial):
    selector_name = trial.suggest_categorical("selector_name", mlp_optuna_seed_space["selector_name"])
    selector_func = selector_name_to_func[selector_name]

    return Pipeline(
        [
            ("fe", FeatureEngineerTransformer(**round4_fe_params)),
            ("geo", GeoTransformer(strategy="drop")),
            ("prep", preprocessor),
            (
                "selector",
                SelectKBest(
                    score_func=selector_func,
                    k=trial.suggest_categorical("k", mlp_optuna_seed_space["k_values"]),
                ),
            ),
            ("scaler", StandardScaler(with_mean=False)),
            (
                "model",
                MLPClassifierWrapper(
                    activation=trial.suggest_categorical("activation", mlp_optuna_seed_space["activation_values"]),
                    hidden_dim=trial.suggest_categorical("hidden_dim", mlp_optuna_seed_space["hidden_dim_values"]),
                    batch_size=trial.suggest_categorical("batch_size", mlp_optuna_seed_space["batch_size_values"]),
                    dropout=suggest_float_or_fixed(
                        trial,
                        "dropout",
                        *mlp_optuna_seed_space["dropout_range"],
                        log=False,
                    ),
                    lr=suggest_float_or_fixed(
                        trial,
                        "lr",
                        *mlp_optuna_seed_space["lr_range"],
                        log=True,
                    ),
                    weight_decay=suggest_float_or_fixed(
                        trial,
                        "weight_decay",
                        *mlp_optuna_seed_space["weight_decay_range"],
                        log=False,
                    ),
                    output_dim=1,
                    max_epochs=80,
                    patience=16,
                    min_delta=1e-3,
                    threshold=0.5,
                    val_size=0.15,
                    random_state=42,
                    verbose=False,
                ),
            ),
        ]
    )


def mlp_objective(trial):
    estimator = build_mlp_optuna_pipeline_from_trial(trial)
    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    row = {
        "trial_id": trial.number + 1,
        "stage": "optuna",
        "model": "MLP",
        **trial.params,
        "pr_auc_mean": cv_res["test_pr_auc"].mean(),
        "roc_auc_mean": cv_res["test_roc_auc"].mean(),
        "recall_mean": cv_res["test_recall"].mean(),
        "precision_mean": cv_res["test_precision"].mean(),
        "f1_mean": cv_res["test_f1"].mean(),
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    }
    mlp_optuna_rows.append(row)
    return row["pr_auc_mean"]


results_mlp_optuna = pd.DataFrame(mlp_optuna_rows)

xgb_optuna_rows = []


def build_xgb_optuna_pipeline_from_trial(trial):
    return Pipeline(
        [
            ("fe", FeatureEngineerTransformer(**round4_fe_params)),
            ("geo", GeoTransformer(strategy="drop")),
            ("prep", preprocessor),
            (
                "model",
                XGBClassifier(
                    objective="binary:logistic",
                    eval_metric="logloss",
                    scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
                    random_state=42,
                    n_jobs=-1,
                    n_estimators=trial.suggest_categorical("n_estimators", xgb_optuna_seed_space["n_estimators_values"]),
                    max_depth=trial.suggest_categorical("max_depth", xgb_optuna_seed_space["max_depth_values"]),
                    min_child_weight=trial.suggest_categorical("min_child_weight", xgb_optuna_seed_space["min_child_weight_values"]),
                    learning_rate=suggest_float_or_fixed(
                        trial,
                        "learning_rate",
                        *xgb_optuna_seed_space["learning_rate_range"],
                        log=True,
                    ),
                    subsample=suggest_float_or_fixed(
                        trial,
                        "subsample",
                        *xgb_optuna_seed_space["subsample_range"],
                        log=False,
                    ),
                    colsample_bytree=suggest_float_or_fixed(
                        trial,
                        "colsample_bytree",
                        *xgb_optuna_seed_space["colsample_bytree_range"],
                        log=False,
                    ),
                    gamma=suggest_float_or_fixed(
                        trial,
                        "gamma",
                        *xgb_optuna_seed_space["gamma_range"],
                        log=False,
                    ),
                    reg_alpha=suggest_float_or_fixed(
                        trial,
                        "reg_alpha",
                        *xgb_optuna_seed_space["reg_alpha_range"],
                        log=False,
                    ),
                    reg_lambda=suggest_float_or_fixed(
                        trial,
                        "reg_lambda",
                        *xgb_optuna_seed_space["reg_lambda_range"],
                        log=True,
                    ),
                ),
            ),
        ]
    )


def xgb_objective(trial):
    estimator = build_xgb_optuna_pipeline_from_trial(trial)
    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    row = {
        "trial_id": trial.number + 1,
        "stage": "optuna",
        "model": "XGBoost",
        **trial.params,
        "pr_auc_mean": cv_res["test_pr_auc"].mean(),
        "roc_auc_mean": cv_res["test_roc_auc"].mean(),
        "recall_mean": cv_res["test_recall"].mean(),
        "precision_mean": cv_res["test_precision"].mean(),
        "f1_mean": cv_res["test_f1"].mean(),
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    }
    xgb_optuna_rows.append(row)
    return row["pr_auc_mean"]


results_xgb_optuna = pd.DataFrame(xgb_optuna_rows)

### ConclusÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â£o

# Persistindo o Melhor Modelo